# ClinicalBERT Fine-Tuning on MACCROBAT

Model: `emilyalsentzer/Bio_ClinicalBERT`  
Dataset: `ktgiahieu/maccrobat2018_2020`

This notebook fine-tunes Bio+Clinical BERT for clinical named entity recognition using MACCROBAT's token-level BIO annotations. It uses a deterministic 80% training and 20% testing split.

## Workflow

1. Download and validate MACCROBAT.
2. Split documents into 80% training and 20% testing data.
3. Build the BIO label mapping.
4. Tokenize long documents into overlapping ClinicalBERT windows.
5. Align word labels with the first subtoken of each word.
6. Fine-tune ClinicalBERT without inspecting the test split.
7. Evaluate once on the held-out test split and run sample inference.

## Imports and Configuration

Select the project's Python kernel before running the notebook. Helper functions come from the CLI script so both workflows use identical preprocessing and metrics.

In [1]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

PROJECT_ROOT = Path.cwd().resolve()
SCRIPT_PATH = Path("models/clinicalbert_maccrobat/train_clinicalbert_maccrobat.py")
while not (PROJECT_ROOT / SCRIPT_PATH).exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Run this notebook from inside Deep-learning-Lab.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.clinicalbert_maccrobat.train_clinicalbert_maccrobat import (
    DEFAULT_DATASET,
    DEFAULT_MODEL,
    build_compute_metrics,
    collect_labels,
    make_splits,
    tokenize_and_align,
    validate_dataset,
)

CONFIG = {
    "model_name": DEFAULT_MODEL,
    "dataset_name": DEFAULT_DATASET,
    "output_dir": PROJECT_ROOT / "outputs/clinicalbert-maccrobat",
    "epochs": 50.0,
    "learning_rate": 2e-5,
    "train_batch_size": 8,
    "eval_batch_size": 8,
    "gradient_accumulation_steps": 1,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_length": 512,
    "stride": 64,
    "test_size": 0.2,
    "seed": 42,
}

set_seed(CONFIG["seed"])
CONFIG

python(53252) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


{'model_name': 'emilyalsentzer/Bio_ClinicalBERT',
 'dataset_name': 'ktgiahieu/maccrobat2018_2020',
 'output_dir': PosixPath('/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/outputs/clinicalbert-maccrobat'),
 'epochs': 50.0,
 'learning_rate': 2e-05,
 'train_batch_size': 8,
 'eval_batch_size': 8,
 'gradient_accumulation_steps': 1,
 'weight_decay': 0.01,
 'warmup_ratio': 0.1,
 'max_length': 512,
 'stride': 64,
 'test_size': 0.2,
 'seed': 42}

## Load and Inspect MACCROBAT

Each clinical document contains parallel `tokens` and `tags` sequences. Every token must have exactly one BIO label.

In [2]:
raw_dataset = load_dataset(CONFIG["dataset_name"], split="train")
validate_dataset(raw_dataset)

print(raw_dataset)
pd.DataFrame({
    "token": raw_dataset[0]["tokens"][:30],
    "tag": raw_dataset[0]["tags"][:30],
})

python(53254) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Dataset({
    features: ['tokens', 'tags'],
    num_rows: 400
})


,token,tag
0,A,O
1,68,B-Age
2,-,I-Age
3,year,I-Age
4,-,I-Age
5,old,I-Age
6,female,B-Sex
7,nonsmoker,B-History
8,",",O
9,nondrinker,B-History


## 80/20 Train and Test Split

Splitting happens at document level before tokenization. This prevents windows from the same clinical document appearing in both sets. The fixed seed makes the split reproducible.

In [3]:
splits = make_splits(
    raw_dataset,
    test_size=CONFIG["test_size"],
    seed=CONFIG["seed"],
)

split_table = pd.DataFrame([
    {
        "split": name,
        "documents": len(dataset),
        "percentage": f"{len(dataset) / len(raw_dataset):.0%}",
    }
    for name, dataset in splits.items()
])
split_table

,split,documents,percentage
0,train,320,80%
1,test,80,20%


## BIO Label Mapping

ClinicalBERT predicts numeric class IDs. These mappings translate between IDs and MACCROBAT labels such as `B-Disease_disorder`, `I-Disease_disorder`, and `O`.

In [4]:
label_names = collect_labels(raw_dataset)
label_to_id = {label: index for index, label in enumerate(label_names)}
id_to_label = {index: label for label, index in label_to_id.items()}

print(f"Number of labels: {len(label_names)}")
pd.DataFrame(label_to_id.items(), columns=["label", "id"])

Number of labels: 82


,label,id
0,O,0
1,B-Activity,1
2,B-Administration,2
3,B-Age,3
4,B-Area,4
...,...,...
77,I-Texture,77
78,I-Therapeutic_procedure,78
79,I-Time,79
80,I-Volume,80


## Tokenization and Label Alignment

Long documents are divided into overlapping windows. Only the first subtoken of each original word receives its BIO label. Special tokens and later subtokens receive `-100`, which excludes them from the loss and metrics.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)
if not tokenizer.is_fast:
    raise ValueError("A fast tokenizer is required for label alignment.")

tokenized_splits = splits.map(
    tokenize_and_align,
    batched=True,
    remove_columns=splits["train"].column_names,
    fn_kwargs={
        "tokenizer": tokenizer,
        "label_to_id": label_to_id,
        "max_length": CONFIG["max_length"],
        "stride": CONFIG["stride"],
    },
    desc="Tokenizing and aligning BIO labels",
)

pd.DataFrame([
    {"split": name, "token_windows": len(dataset)}
    for name, dataset in tokenized_splits.items()
])

,split,token_windows
0,train,620
1,test,160


In [6]:
example = tokenized_splits["train"][0]
example_tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
example_labels = [
    id_to_label[label_id] if label_id != -100 else "IGNORED"
    for label_id in example["labels"]
]
pd.DataFrame({
    "model_token": example_tokens,
    "training_label": example_labels,
}).head(40)

,model_token,training_label
0,[CLS],IGNORED
1,a,O
2,13,B-Age
3,year,I-Age
4,-,I-Age
5,old,I-Age
6,boy,B-Sex
7,presented,B-Clinical_event
8,with,O
9,a,O


## Create ClinicalBERT for Token Classification

The pretrained Bio+Clinical BERT encoder is reused and a new token-classification layer is initialized. Warnings about newly initialized classifier weights are expected because MACCROBAT training teaches this new layer.

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
)

print(f"Loaded {CONFIG['model_name']} with {model.config.num_labels} labels.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture

Loaded emilyalsentzer/Bio_ClinicalBERT with 82 labels.


## Configure Training

The held-out test set is deliberately not passed to the Trainer here. Checkpoints are saved after each epoch, while test evaluation happens only after training.

In [8]:
steps_per_epoch = math.ceil(
    len(tokenized_splits["train"])
    / (CONFIG["train_batch_size"] * CONFIG["gradient_accumulation_steps"])
)
warmup_steps = round(
    steps_per_epoch * CONFIG["epochs"] * CONFIG["warmup_ratio"]
)

training_args = TrainingArguments(
    output_dir=str(CONFIG["output_dir"]),
    num_train_epochs=CONFIG["epochs"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=warmup_steps,
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    save_total_limit=2,
    report_to="none",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=build_compute_metrics(id_to_label),
)

print(f"Training windows: {len(tokenized_splits['train'])}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Warmup steps: {warmup_steps}")

Training windows: 620
Steps per epoch: 78
Warmup steps: 390


## Train ClinicalBERT

This is the computationally expensive cell. On a compatible CUDA GPU, add `fp16=True` to `TrainingArguments`. Reduce the batch size if GPU memory is limited.

In [9]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(CONFIG["output_dir"])
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
train_result.metrics

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,4.429110
20,4.355273
30,4.207793
40,3.967082
50,3.536765
60,2.801060
70,2.042288
80,1.530216
90,1.593480
100,1.654579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =              50.0
  total_flos               =         7547127GF
  train_loss               =            0.2412
  train_runtime            = 1 day, 0:01:04.89
  train_samples_per_second =             0.359
  train_steps_per_second   =             0.045


{'train_runtime': 86464.8934,
 'train_samples_per_second': 0.359,
 'train_steps_per_second': 0.045,
 'total_flos': 8103665945449968.0,
 'train_loss': 0.24115930008964662,
 'epoch': 50.0}

## Final Test Evaluation

Run this cell once after training. It reports entity-level precision, recall, and F1 from `seqeval`, along with token accuracy and test loss.

In [10]:
print(f"Evaluating on held-out 20% test split: {len(tokenized_splits['test'])} token windows")

test_metrics = trainer.evaluate(
    eval_dataset=tokenized_splits["test"],
    metric_key_prefix="test",
)
trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)

test_summary = pd.Series({
    "test_loss": test_metrics.get("test_loss"),
    "test_precision": test_metrics.get("test_precision"),
    "test_recall": test_metrics.get("test_recall"),
    "test_f1": test_metrics.get("test_f1"),
    "test_accuracy": test_metrics.get("test_accuracy"),
}).to_frame("value")

test_summary

Evaluating on held-out 20% test split: 160 token windows


/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
0.010250,0.341472,3900,0.828508,0.928420,0.875624,0.955046


***** test metrics *****
  test_accuracy  =  0.955
  test_f1        = 0.8756
  test_loss      = 0.3415
  test_precision = 0.8285
  test_recall    = 0.9284


,value
test_loss,0.341472
test_precision,0.828508
test_recall,0.928420
test_f1,0.875624
test_accuracy,0.955046


## Sample Clinical NER

The pipeline groups BIO token predictions into readable entity spans. Change `input_text` to try another clinical sentence.

In [11]:
input_text = (
    "A 67-year-old woman with hypertension presented with severe chest pain. "
    "She was given aspirin 325 mg and underwent coronary angiography."
)

ner_pipeline = pipeline(
    task="token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)
predictions = ner_pipeline(input_text)

pd.DataFrame([
    {
        "entity": item["entity_group"],
        "text": input_text[item["start"]:item["end"]],
        "start": item["start"],
        "end": item["end"],
        "score": round(float(item["score"]), 4),
    }
    for item in predictions
])

,entity,text,start,end,score
0,Age,67-year-old,2,13,0.9987
1,Sex,woman,14,19,0.9966
2,History,hypertens,25,34,0.5684
3,Disease_disorder,ion,34,37,0.3318
4,Severity,severe,53,59,0.6066
5,Sign_symptom,chest,60,65,0.5819
6,Sign_symptom,pain,66,70,0.7575
7,Medication,aspirin,86,93,0.8436
8,Dosage,325 mg,94,100,0.8951
9,Diagnostic_procedure,angiography,124,135,0.8282
